# Iris Classification Walkthrough

This notebook walks you through a simple, end-to-end ML workflow on the Iris dataset:
- Load data, explore basics
- Train/test split and scaling
- Train Logistic Regression
- Evaluate and visualize results
- Save and reuse the model

Run cells top-to-bottom. Make sure you've installed `requirements.txt` in your virtual environment.


In [ ]:
# Imports
import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Load dataset
iris = datasets.load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name='target')

# Quick look
display(X.head())
print('Class distribution:', y.value_counts().sort_index().to_dict())

# Visualize pairplot (may take a moment)
sns.pairplot(pd.concat([X, y], axis=1), hue='target', diag_kind='hist')
plt.show()

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(
    X.values, y.values, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Train model
model = LogisticRegression(max_iter=500)
model.fit(X_train_s, y_train)

# Evaluate
y_pred = model.predict(X_test_s)
print('Accuracy:', round(accuracy_score(y_test, y_pred), 4))
print('\nClassification Report\n', classification_report(y_test, y_pred))
print('Confusion Matrix\n', confusion_matrix(y_test, y_pred))


In [ ]:
# Compare multiple models
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

models = {
    'LogReg': LogisticRegression(max_iter=500),
    'SVC': SVC(),
    'RF': RandomForestClassifier(random_state=42),
    'KNN': KNeighborsClassifier()
}

results = []
for name, clf in models.items():
    clf.fit(X_train_s, y_train)
    y_pred_m = clf.predict(X_test_s)
    acc_m = accuracy_score(y_test, y_pred_m)
    results.append({'model': name, 'accuracy': acc_m})

pd.DataFrame(results).sort_values('accuracy', ascending=False)


In [ ]:
# Confusion matrix heatmap for best model (RandomForest as example)
from sklearn.metrics import confusion_matrix
best_clf = RandomForestClassifier(random_state=42)
best_clf.fit(X_train_s, y_train)
y_pred_best = best_clf.predict(X_test_s)
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(4,3))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix (RandomForest)')
plt.tight_layout()
plt.show()
